# Introduction to Simple IFRS17 Reporting

In this notebook we will:
1. Generate dummy term life insurance
2. Calculate BEL, RA, and CSM
3. Aggregate the Contract based on profitability, and cohort. We assume all has the same risk since the contract is the same

# Packages

In [ ]:
import numpy as np
import pandas as pd

# Generate Dummy Term Life Insurance

In [ ]:
def gen_data(insurance_start_date, insurance_coverage, sum_assured, age_range, n_contracts, seed):
    """
    This is a function to create dummy data on a term life insurance

    Parameter:
    ==================
    1. insurance_start_date: insurance first issued: String with "dd-mm-yyyy" format
    2. insurance_coverage: how long the insurance cover: integer
    3. sum_assured: how much the insurance liable to pay: integer
    4. age_range: who is eligible to apply this product: array[2] [min_age, max_age]
    5. n_contracts: number of data generated: integer
    6. seed: to fixed the output: integer

    Return:
    ===========
    Pandas dataframe with following columns:
    | ID | DoB | Gender | Ins_Start | Ins_End | SA | Premium | 
    """
    ID = np.random.choice(range(n_contracts), size=n_contracts, replace=False)
    Ins_Start = pd.to_datetime(insurance_start_date, format="%d-%m-%Y") + pd.to_timedelta(np.random.randint(0, 364, size=n_contracts), unit='D')
    Ins_End = Ins_Start + pd.DateOffset(years=insurance_coverage)
    DoB = pd.to_datetime({
        'year': Ins_Start.year - np.random.randint(age_range[0], age_range[1], size=n_contracts),
        'month': Ins_Start.month,
        'day': Ins_Start.day
    })
    Gender = np.random.choice(['Male', 'Female'], size=n_contracts)
    SA = 100_000 * np.random.randint(1, 20, n_contracts)
    Premium = SA * 0.01
    data = pd.DataFrame({'ID': ID, 
                         'DoB': DoB, 
                         'Gender': Gender,
                         'Ins_Start': Ins_Start,
                         'Ins_End': Ins_End,
                         'SA': SA,
                         'Premium': Premium})
    return data

In [ ]:
data_2023 = gen_data('31-12-2022', 5, 100, [20,55], 100000, 100000)
data_2023.head()

In [ ]:
pd.DataFrame(((pd.date_range(start=data_2023['Ins_Start'][0] + pd.offsets.MonthEnd(0), end=data_2023['Ins_End'][0] +  pd.offsets.MonthEnd(0), freq='M') - data_2023['DoB'][0]).days/365.25))

In [ ]:
def actuarial_table(data):
    valuation_date = pd.date_range(start=data['Ins_Start'] + pd.offsets.MonthEnd(0), end=data['Ins_End'] +  pd.offsets.MonthEnd(0), freq='M')
    age = (valuation_date - data['DoB']).days/365.25 
    return valuation_date, age

In [ ]:
actuarial_table(data_2023.iloc[0,])

In [ ]:
def profit_tagging(data):
    """
    This is a function to tag profitability

    Parameter:
    ==================
    1. data: insurance data: pandas format format
    2. insurance_coverage: how long the insurance cover: integer
    3. sum_assured: how much the insurance liable to pay: integer
    4. age_range: who is eligible to apply this product: array[2] [min_age, max_age]
    5. n_contracts: number of data generated: integer
    6. seed: to fixed the output: integer

    Return:
    ===========
    Pandas dataframe with following columns:
    | ID | DoB | Gender | Ins_Start | Ins_End | SA | Premium | 
    """